# Agents SDK guardrails and structured outputs

## Learning goals

- Define a typed final output with Pydantic
- Distinguish structural validation from guardrail policy
- Implement deterministic input and output guardrails
- Test guardrails offline and understand tripwire exceptions
- Know where agent-level and tool-level guardrails apply


## Two complementary boundaries

`output_type` tells the SDK the required final data shape. It is ideal for fields, enums, ranges, and nested structures. Guardrails evaluate policy around the run: whether input is allowed, whether final output is acceptable, or whether a function-tool call should proceed.

A valid Pydantic object can still violate policy, and a guardrail pass does not guarantee the output matches a consumer schema. Use both when both concerns exist.


## Mental model

```mermaid
flowchart LR
  A[User input] --> B[Input guardrails]
  B -->|tripwire| C[InputGuardrailTripwireTriggered]
  B -->|pass| D[Runner and agent loop]
  D --> E[Typed final candidate]
  E --> F[Pydantic output_type validation]
  F --> G[Output guardrails]
  G -->|tripwire| H[OutputGuardrailTripwireTriggered]
  G -->|pass| I[Typed final_output]
  D --> J[Function tool call]
  J --> K[Tool input and output guardrails]
```

Agent input guardrails apply at the start of a chain; agent output guardrails apply to the final agent. Function-tool guardrails wrap each decorated function-tool call and are a separate control surface.


## Define the final contract

Keep the schema narrow enough for consumers to trust. `extra="forbid"` catches unexpected fields, and nested day plans make the itinerary structure explicit.


In [ ]:
from typing import Literal

from pydantic import BaseModel, ConfigDict, Field


class DayPlan(BaseModel):
    model_config = ConfigDict(extra="forbid")
    day: int = Field(ge=1, le=14)
    activities: list[str] = Field(min_length=1, max_length=6)
    notes: str = Field(max_length=300)


class Itinerary(BaseModel):
    model_config = ConfigDict(extra="forbid")
    city: str = Field(min_length=2, max_length=100)
    days: list[DayPlan] = Field(min_length=1, max_length=14)
    estimated_total: int = Field(ge=0)
    currency: Literal["INR", "USD"]
    assumptions: list[str] = Field(min_length=1, max_length=8)


print(Itinerary.model_json_schema())


## Guardrails can use trusted application context

The user's authenticated limits belong in local context, not model-generated output. Here, an output guardrail compares the typed estimate with an application-provided budget ceiling.


In [ ]:
from dataclasses import dataclass


@dataclass(frozen=True)
class PlanningContext:
    max_budget: int
    currency: Literal["INR", "USD"]


planning_context = PlanningContext(max_budget=20_000, currency="INR")
print(planning_context)


## Deterministic input guardrail

This guardrail rejects impossible input sizes and requests to complete a booking/payment. It does not attempt to detect every malicious phrase. `run_in_parallel=False` makes the guardrail complete before the main agent starts, which avoids spending a model call on input we know must be blocked.


In [ ]:
import json
from typing import Any

from agents import (
    Agent,
    GuardrailFunctionOutput,
    RunContextWrapper,
    input_guardrail,
)


def input_as_text(value: str | list[dict[str, Any]]) -> str:
    return value if isinstance(value, str) else json.dumps(value, default=str)


@input_guardrail(run_in_parallel=False)
def travel_request_guardrail(
    ctx: RunContextWrapper[PlanningContext],
    agent: Agent,
    input: str | list[dict[str, Any]],
) -> GuardrailFunctionOutput:
    text = input_as_text(input).strip()
    too_long = len(text) > 4_000
    requests_transaction = any(
        phrase in text.lower()
        for phrase in ["book and pay", "complete the payment", "purchase the ticket"]
    )
    reason = "input_too_long" if too_long else "transaction_not_supported" if requests_transaction else "allowed"
    return GuardrailFunctionOutput(
        output_info={"reason": reason, "character_count": len(text)},
        tripwire_triggered=too_long or requests_transaction,
    )


## Semantic output guardrail

Pydantic proves that `estimated_total` is a non-negative integer. The guardrail enforces that it stays within this user's budget and uses the expected currency.


In [ ]:
from agents import output_guardrail


@output_guardrail
def budget_output_guardrail(
    ctx: RunContextWrapper[PlanningContext],
    agent: Agent,
    output: Itinerary,
) -> GuardrailFunctionOutput:
    wrong_currency = output.currency != ctx.context.currency
    over_budget = output.estimated_total > ctx.context.max_budget
    return GuardrailFunctionOutput(
        output_info={
            "over_budget": over_budget,
            "wrong_currency": wrong_currency,
        },
        tripwire_triggered=over_budget or wrong_currency,
    )


## Attach output type and guardrails to the agent

The final `RunResult.final_output` will be an `Itinerary` when this agent finishes successfully. Handoffs can change the finishing agent and therefore the possible output type.


In [ ]:
import os

from agents import ModelSettings

planner_agent = Agent[PlanningContext](
    name="Guarded itinerary planner",
    instructions=(
        "Return a practical itinerary matching the requested budget and currency. "
        "List assumptions and never claim a booking or payment completed."
    ),
    model=os.getenv("OPENAI_MODEL", "gpt-4.1-mini"),
    model_settings=ModelSettings(max_tokens=600, store=False),
    output_type=Itinerary,
    input_guardrails=[travel_request_guardrail],
    output_guardrails=[budget_output_guardrail],
)

print("Output type:", planner_agent.output_type)
print("Input guardrails:", len(planner_agent.input_guardrails))
print("Output guardrails:", len(planner_agent.output_guardrails))


## Test guardrail functions offline

Calling each guardrail's `run` method verifies the application logic without invoking a model. During a real `Runner` run, a true tripwire raises the corresponding SDK exception and halts the run.


In [ ]:
context_wrapper = RunContextWrapper(context=planning_context)

allowed_input = await travel_request_guardrail.run(
    planner_agent,
    "Plan two days in Jaipur within INR 20,000.",
    context_wrapper,
)
blocked_input = await travel_request_guardrail.run(
    planner_agent,
    "Book and pay for my hotel.",
    context_wrapper,
)

sample_itinerary = Itinerary(
    city="Jaipur",
    days=[DayPlan(day=1, activities=["City Palace"], notes="Allow a relaxed morning.")],
    estimated_total=18_000,
    currency="INR",
    assumptions=["Excludes travel to Jaipur"],
)
output_check = await budget_output_guardrail.run(
    context_wrapper,
    planner_agent,
    sample_itinerary,
)

print("Allowed input tripwire:", allowed_input.output.tripwire_triggered)
print("Blocked input tripwire:", blocked_input.output.tripwire_triggered)
print("Output tripwire:", output_check.output.tripwire_triggered)
assert not allowed_input.output.tripwire_triggered
assert blocked_input.output.tripwire_triggered
assert not output_check.output.tripwire_triggered


## Optional live run and tripwire handling

Catch the specific tripwire exceptions at the application boundary. Return a safe product message and retain only allowlisted diagnostics.


In [ ]:
from agents import (
    InputGuardrailTripwireTriggered,
    OutputGuardrailTripwireTriggered,
    Runner,
)

RUN_LIVE_CALL = False

if RUN_LIVE_CALL:
    try:
        result = await Runner.run(
            planner_agent,
            "Plan a relaxed day in Jaipur within INR 20,000.",
            context=planning_context,
            max_turns=5,
        )
        itinerary: Itinerary = result.final_output
        print(itinerary.model_dump(mode="json"))
    except InputGuardrailTripwireTriggered:
        print("The request is outside the supported input policy.")
    except OutputGuardrailTripwireTriggered:
        print("The generated itinerary did not satisfy the output policy.")
else:
    print("Live guarded run skipped.")


## Guardrail placement matters

- Input guardrails attached to an agent run only when that agent is first in the chain.
- Output guardrails attached to an agent run only when that agent produces the final output.
- Function-tool input/output guardrails run around each decorated function tool.
- Handoffs and hosted tools have different pipelines; do not assume function-tool guardrails cover them.
- Parallel input guardrails may run alongside the first model call; use `run_in_parallel=False` when preflight blocking must happen first.
- Guardrail metadata can appear in run results/traces; avoid placing secrets in `output_info`.


## Failure cases and improvements

- **Schema accepts semantically bad plan:** add domain guardrails and evaluation.
- **Guardrail uses another model for a simple length check:** prefer deterministic preflight checks.
- **Parallel guardrail still incurs a model call:** set `run_in_parallel=False` for hard preflight gates.
- **Tripwire exception becomes a 500:** catch specific SDK exceptions at the app boundary.
- **Only first agent is guarded:** map controls across handoffs and tools explicitly.
- **Guardrail logs full sensitive input:** retain reason codes and safe measurements instead.


## Exercise

1. Add an output guardrail that requires one rest break for itineraries longer than two days.
2. Test an over-budget itinerary and inspect the tripwire metadata.
3. Add a deterministic empty-input guardrail.
4. Design a function-tool guardrail for a write action.
5. Draw which controls run before, during, and after a handoff chain.


## Official references

- [Agents SDK guardrails](https://openai.github.io/openai-agents-python/guardrails/)
- [Agent output types](https://openai.github.io/openai-agents-python/agents/)
- [Guardrail API reference](https://openai.github.io/openai-agents-python/ref/guardrail/)


## Summary

Use Pydantic `output_type` for the final data shape and guardrails for input, output, or function-tool policy. Test deterministic checks offline, choose parallel versus preflight execution deliberately, catch tripwire exceptions at the application boundary, and map guardrail coverage across tools and handoffs rather than assuming one check protects the whole system.
